# 03 · Graph Classification — GIN on MUTAG

> 🎬 **Watch first:** [Batching graphs in PyG](https://youtu.be/bD7IDjEJ16M) ·
> [Why sum beats mean — GIN & the WL test](https://youtu.be/sJ-m8AIJC-s)

**Task:** classify *whole graphs*. Each graph is a molecule; predict whether it's mutagenic. Two new ideas vs.
node classification:

1. **Mini-batching graphs** — many small graphs are glued into one big *disconnected* graph; a `batch` vector
   records which graph each node belongs to. `DataLoader` does this automatically.
2. **Readout / global pooling** — collapse node embeddings into **one vector per graph** (`global_add_pool`).

**Model — GIN.** The most *expressive* message-passing GNN: its power matches the **Weisfeiler–Lehman** graph
isomorphism test. Update rule:

$$ h_v' = \text{MLP}\Big( (1+\epsilon)\, h_v + \sum_{u\in N(v)} h_u \Big) $$

**Source:** Xu et al., *How Powerful are Graph Neural Networks?*, ICLR 2019
([arXiv:1810.00826](https://arxiv.org/abs/1810.00826)). Dataset: Morris et al., TUDataset 2020
([arXiv:2007.08663](https://arxiv.org/abs/2007.08663)).

In [1]:
import sys, os
ROOT = os.getcwd()
if os.path.basename(ROOT) == "notebooks":
    ROOT = os.path.dirname(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")  # safe Apple-GPU fallback

import torch
from utils.device import get_device, device_report
device = get_device()
print(device_report())


Selected device : MPS
CUDA GPU        : not available
Apple MPS (Metal): available
CPU threads     : 4


In [2]:
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader

dataset = TUDataset(root=os.path.join(ROOT, "03_graph_classification", "data"), name="MUTAG").shuffle()
print(len(dataset), "graphs |", dataset.num_features, "node features |", dataset.num_classes, "classes")
n = int(len(dataset)*0.8)
train_loader = DataLoader(dataset[:n], batch_size=32, shuffle=True)
test_loader  = DataLoader(dataset[n:], batch_size=32)

b = next(iter(train_loader))
print(f"one batch -> {b.num_graphs} graphs glued into {b.num_nodes} nodes; batch vec {tuple(b.batch.shape)}")

188 graphs | 7 node features | 2 classes
one batch -> 32 graphs glued into 564 nodes; batch vec (564,)


In [3]:
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, BatchNorm1d
from torch_geometric.nn import GINConv, global_add_pool

class GIN(torch.nn.Module):
    def __init__(self, in_dim, hidden, n_cls, layers=3):
        super().__init__()
        self.convs = torch.nn.ModuleList()
        dims = [in_dim] + [hidden]*layers
        for i in range(layers):
            mlp = Sequential(Linear(dims[i], hidden), BatchNorm1d(hidden), ReLU(),
                             Linear(hidden, hidden), ReLU())
            self.convs.append(GINConv(mlp, train_eps=True))
        self.lin1 = Linear(hidden, hidden); self.lin2 = Linear(hidden, n_cls)
    def forward(self, x, edge_index, batch):
        for c in self.convs: x = c(x, edge_index)
        x = global_add_pool(x, batch)            # READOUT: one vector per graph
        x = F.relu(self.lin1(x)); x = F.dropout(x, 0.5, training=self.training)
        return self.lin2(x)

model = GIN(dataset.num_features, 32, dataset.num_classes).to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.01)

@torch.no_grad()
def test(loader):
    model.eval(); correct = total = 0
    for d in loader:
        d = d.to(device)
        correct += int((model(d.x, d.edge_index, d.batch).argmax(1) == d.y).sum()); total += d.num_graphs
    return correct/total

for epoch in range(1, 101):
    model.train()
    for d in train_loader:
        d = d.to(device); opt.zero_grad()
        F.cross_entropy(model(d.x, d.edge_index, d.batch), d.y).backward(); opt.step()
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} | train {test(train_loader):.3f} | test {test(test_loader):.3f}")
print("final test acc:", round(test(test_loader), 3))

epoch  20 | train 0.640 | test 0.553


epoch  40 | train 0.907 | test 0.842


epoch  60 | train 0.933 | test 0.895


epoch  80 | train 0.967 | test 0.895


epoch 100 | train 0.867 | test 0.816
final test acc: 0.816


## ✅ Notes
- MUTAG has only 188 graphs, so accuracy is noisy run-to-run (expect ~75–90%).
- Swap `global_add_pool` → `global_mean_pool` and compare; **sum** preserves more info (GIN's choice).
- CLI version with confusion matrix + checkpoints: `python project/main.py train --dataset MUTAG --model gin`.

➡ Next: **04 · Link Prediction**.